# 02 — Raster mosaicking and feature-stack assembly

Merges exported Sentinel-2 tiles, extracts and renames single bands, and assembles the per-mosaic multiband feature stack (spectral bands, indices, GLCM textures, phenology, terrain) that the classifier reads.

Set `DATA_DIR` to your project data root. Paths below are relative to it.

_Manuscript: Methods — Feature stack construction._

### Merging

In [ ]:
import os
import rasterio
from rasterio.merge import merge
from glob import glob

# === Set directory containing the downloaded tiles ===
input_dir = r"DATA_DIR\Combined\2025\Rasters\Mosaic2"  
output_dir = os.path.join(input_dir, "merged")
os.makedirs(output_dir, exist_ok=True)

# === Features to merge ===
features = ['Blue', 'Green', 'Red', 'NIR', 'NDVI', 'NARI']

for feature in features:
    print(f"Merging tiles for: {feature}")
    
    # Find all matching files
    pattern = f"*{feature}_Stack_*.tif"
    file_list = glob(os.path.join(input_dir, pattern))
    
    if not file_list:
        print(f"❌ No files found for {feature}")
        continue
    
    # Open all files
    src_files_to_mosaic = [rasterio.open(fp) for fp in file_list]
    
    try:
        # Merge
        mosaic, out_trans = merge(src_files_to_mosaic)
        
        # Use metadata from the first tile
        out_meta = src_files_to_mosaic[0].meta.copy()
        out_meta.update({
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": out_trans
        })

        # Output file path
        out_fp = os.path.join(output_dir, f"Merged_{feature}_Stack_2024_Filled.tif")

        # Write to disk
        with rasterio.open(out_fp, "w", **out_meta) as dest:
            dest.write(mosaic)

        print(f"✅ Saved: {out_fp}")

    finally:
        # Close all source files so they are not locked
        for src in src_files_to_mosaic:
            src.close()

print("✅ All merging done.")



In [ ]:
import os
import numpy as np
import rasterio

input_dir = r"DATA_DIR\Combined\2018\Rasters\Mosaic1"
band_files = [os.path.join(input_dir, f"NIR_Stack_2017_B{i}.tif") for i in range(1, 5+1)]
output_fp = os.path.join(input_dir, "Merged_NIR_Stack_2017_Filled.tif")

# Use band 1 as reference
with rasterio.open(band_files[0]) as ref:
    profile = ref.profile.copy()

# Keep original dtype if you can (UInt16 is compact). Use NaN only if float.
dtype = profile["dtype"]  # likely 'uint16'
nodata = profile.get("nodata", 0)

# Update profile for a robust, >4GB-safe GTiff
profile.update(
    count=len(band_files),
    compress="DEFLATE",       # or "LZW" or "ZSTD" if GDAL supports it
    predictor=2,              # better compression for continuous data
    tiled=True,
    blockxsize=512,
    blockysize=512,
    BIGTIFF="YES",            # <-- the key to break the 4GB ceiling
)

with rasterio.open(output_fp, "w", **profile) as dst:
    for i, fp in enumerate(band_files, start=1):
        with rasterio.open(fp) as src:
            arr = src.read(1)
            # Respect input mask/nodata
            m = src.read_masks(1)  # 0=nodata
            if dtype.startswith("float"):
                arr = arr.astype("float32")
                arr[m == 0] = np.nan
            else:
                arr = arr.astype(dtype)
                arr[m == 0] = nodata if nodata is not None else 0
        dst.write(arr, i)
        dst.set_band_description(i, f"NIR_B{i}")

print("✅ TIFF written:", output_fp)



### Extracting single bands

In [ ]:
import os
import rasterio

# === Setup ===
input_dir = r"DATA_DIR\Combined\2025\Rasters\Mosaic2\merged"  # Folder with merged feature stacks
output_dir = r"DATA_DIR\Combined\2025\Rasters\Mosaic2\bands"  # Output folder for individual bands
os.makedirs(output_dir, exist_ok=True)

# === List of features to extract ===
features = ['Blue', 'Green', 'Red', 'NIR', 'NDVI', 'NARI']
year = 2024  # Or adjust to match filenames

# === Loop over each feature stack ===
for feature in features:
    input_file = os.path.join(input_dir, f"Merged_{feature}_Stack_{year}_Filled.tif")
    
    if not os.path.exists(input_file):
        print(f"❌ File not found: {input_file}")
        continue

    with rasterio.open(input_file) as src:
        print(f"📂 Processing: {os.path.basename(input_file)} with {src.count} bands")
        
        for band_idx in range(1, src.count + 1):
            out_name = f"{feature}_Stack_{year}_B{band_idx}.tif"
            out_path = os.path.join(output_dir, out_name)

            meta = src.meta.copy()
            meta.update(count=1)

            with rasterio.open(out_path, "w", **meta) as dst:
                dst.write(src.read(band_idx, out_dtype="float32"), 1)
                dst.set_band_description(1, f"{feature}_B{band_idx}")
            
            print(f"✅ Extracted: {out_name}")

print("🎉 All bands extracted.")


### Renaming and Organizing

In [ ]:
import os
import glob
import joblib
import re
import shutil

# === INPUTS ===
source_folder = r"DATA_DIR\Combined\2025\Rasters\Mosaic2\bands"
destination_folder = r"DATA_DIR\Combined\2025\Rasters\Mosaic2\selected"
model_path = r"DATA_DIR\Combined\5\Model\training_data_S2_selected30_merged.pkl"
os.makedirs(destination_folder, exist_ok=True)

# === Load model feature names
model = joblib.load(model_path)
expected_features = list(model.feature_names_in_)

# === Get all available raster files in source
existing_files = glob.glob(os.path.join(source_folder, "*.tif"))
existing_map = {os.path.basename(f).replace(".tif", ""): f for f in existing_files}

# === Function to extract and modify file name for matching and renaming
def extract_pattern(name, new_year="2024"):
    # Handle the pattern *Stack_2017_B* (without Sentinel2_ or Filled_)
    match = re.match(r"(.*_Stack)_(\d{4})?_?B(\d+)", name)
    if match:
        # Update the year to 2024 and add Sentinel2_ and Filled_ prefix
        return f"Sentinel2_{match.group(1)}_{new_year}_Filled_B{match.group(3)}"
    return name

# === Matching and copying files
used_files = set()
copied = 0

for feature in expected_features:
    pattern = extract_pattern(feature)

    # Find the matching file
    candidates = [
        f for f in existing_map
        if extract_pattern(f) == pattern and f not in used_files
    ]

    if candidates:
        original = candidates[0]
        used_files.add(original)

        src_path = existing_map[original]
        dst_path = os.path.join(destination_folder, feature + ".tif")

        shutil.copy2(src_path, dst_path)
        print(f"📁 Copied and renamed: {original}.tif → {feature}.tif")
        copied += 1
    else:
        print(f"❌ No match for: {feature}")

print(f"\n✅ Total copied and renamed: {copied} / {len(expected_features)}")


In [ ]:
import os
import glob
import rasterio
import numpy as np
import joblib
from rasterio.enums import Resampling
from rasterio.warp import reproject, calculate_default_transform

# === Define Paths ===
raster_folder = r"DATA_DIR\Combined\2025\Rasters\Mosaic2\selected"
output_stack = r"DATA_DIR\Combined\2025\Rasters\raster_stack_S2_selected30_mosaic2.tif"
reference_raster_path = r"DATA_DIR\Combined\2025\Rasters\Mosaic2\selected\DTM_resampled.tif"

# === Optional: Provide model path to enforce feature order ===
model_path = r"DATA_DIR\Combined\5\Model\training_data_S2_selected30_merged.pkl"  # or None

# === Load Reference Raster ===
with rasterio.open(reference_raster_path) as ref:
    ref_meta = ref.meta.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_width, ref_height = ref.width, ref.height
    ref_res = ref.res
    ref_bounds = ref.bounds

# === Determine Raster Order ===
raster_files = glob.glob(os.path.join(raster_folder, "*.tif"))
file_map = {
    os.path.basename(fp).replace(".tif", ""): fp
    for fp in raster_files
}

if model_path and os.path.exists(model_path):
    model = joblib.load(model_path)
    expected_order = list(model.feature_names_in_)

    ordered_files = []
    missing = []

    for feat in expected_order:
        if feat in file_map:
            ordered_files.append(file_map[feat])
        else:
            missing.append(feat)

    if missing:
        raise FileNotFoundError(f"❌ Missing raster files for: {missing}")
    
    raster_files = ordered_files
    print("✅ Raster stacking order based on model features.")
else:
    raster_files = sorted(raster_files)
    print("⚠️ No model provided or path invalid — using alphabetical order.")

# === Process and Stack Bands ===
all_bands = []
band_names = []

for raster_path in raster_files:
    base_name = os.path.basename(raster_path).replace(".tif", "")
    print(f"📂 Processing: {raster_path}")

    with rasterio.open(raster_path) as src:
        src_crs = src.crs if src.crs is not None else ref_crs
        num_bands = src.count

        same_extent = src.bounds == ref_bounds
        same_res = np.allclose(src.res, ref_res, atol=1e-6)

        for band_idx in range(1, num_bands + 1):
            print(f"  📊 Reading Band {band_idx}/{num_bands} from {base_name}")
            data = src.read(band_idx)

            if not (same_extent and same_res):
                print(f"  🔄 Aligning & Resampling Band {band_idx} to match reference raster...")
                transform, width, height = calculate_default_transform(
                    src_crs, ref_crs, ref_width, ref_height, *ref_bounds
                )
                aligned_data = np.empty((ref_height, ref_width), dtype=np.float32)
                reproject(
                    source=data,
                    destination=aligned_data,
                    src_transform=src.transform,
                    src_crs=src_crs,
                    dst_transform=transform,
                    dst_crs=ref_crs,
                    resampling=Resampling.bilinear
                )
            else:
                aligned_data = data

            all_bands.append(aligned_data.astype(np.float32))
            band_names.append(f"{base_name}_B{band_idx}" if num_bands > 1 else base_name)

# === Save Final Stack ===
stacked_array = np.stack(all_bands, axis=0)

ref_meta.update({
    "count": stacked_array.shape[0],
    "dtype": "float32",
    "compress": "lzw",
    "BIGTIFF": "YES",
    "tiled": True,
})

with rasterio.open(output_stack, "w", **ref_meta) as dst:
    dst.write(stacked_array)
    for idx, band_name in enumerate(band_names):
        dst.set_band_description(idx + 1, band_name)

print(f"\n✅ Raster stack saved at:\n{output_stack}")


In [ ]:
import rasterio
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

# Define paths
raster_stack_path = r"DATA_DIR\Combined\2024\Rasters\Mosaic1\raster_stack_S2_selected30_mosaic1_normalized.tif"
model_path = r"DATA_DIR\Combined\2024\Model\training_data_S2_selected30_merged_normalized_2024.pkl"
output_raster = r"DATA_DIR\Combined\2024\Results\Pinus_mugo_classification_selected30_mosaic1_new_normalized.tif"


# Load trained model
rf_model = joblib.load(model_path)
print("✅ Loaded trained Random Forest model.")

# Expected band order from model
expected_band_names = list(rf_model.feature_names_in_)

# Open raster stack
with rasterio.open(raster_stack_path) as src:
    meta = src.meta.copy()
    band_count = src.count
    height, width = src.height, src.width

    # Map raster band descriptions to indices
    raster_band_names = list(src.descriptions)
    name_to_index = {desc: i for i, desc in enumerate(raster_band_names)}

    # Check and warn for mismatches
    mismatched = [b for b in expected_band_names if b not in name_to_index]
    if mismatched:
        raise ValueError(f"❌ Missing bands in raster: {mismatched}")

    # Get band indices in model-required order
    ordered_indices = [name_to_index[name] for name in expected_band_names]

    # Update metadata for output raster
    meta.update({
        "count": 1,
        "dtype": "uint8",
        "nodata": 0,
        "compress": "lzw",
        "BIGTIFF": "YES",
        "tiled": True
    })

    # Tile size for memory-efficient processing
    tile_size = 512

    with rasterio.open(output_raster, "w", **meta) as dst:
        for i in range(0, height, tile_size):
            for j in range(0, width, tile_size):
                window = rasterio.windows.Window(
                    j, i, min(tile_size, width - j), min(tile_size, height - i)
                )

                # Read and reorder bands based on expected model input
                raster_data = np.stack([
                    src.read(b + 1, window=window) for b in ordered_indices
                ]).astype(np.float32)  # Shape: (bands, H, W)

                tile_height, tile_width = raster_data.shape[1], raster_data.shape[2]
                reshaped = raster_data.reshape(len(ordered_indices), -1).T  # (N_pixels, N_bands)

                # Predict only on valid pixels
                invalid_mask = ~np.isfinite(reshaped).all(axis=1)
                predictions = np.zeros((reshaped.shape[0],), dtype=np.uint8)

                if (~invalid_mask).sum() > 0:
                    valid_data = reshaped[~invalid_mask]
                    predictions[~invalid_mask] = rf_model.predict(valid_data).astype(np.uint8)

                classified_tile = predictions.reshape(tile_height, tile_width)
                dst.write(classified_tile, 1, window=window)

print(f"✅ Classification map saved: {output_raster}")


In [ ]:
import rasterio
from rasterio.merge import merge
import os

# 📂 Input raster file paths
raster1 = r"DATA_DIR\Combined\2021\Rasters\Mosaic1\Pinus_mugo_classification_selected30_mosaic1.tif"
raster2 = r"DATA_DIR\Combined\2021\Rasters\Mosaic2\Pinus_mugo_classification_selected30_mosaic2.tif"
output_path = r"DATA_DIR\Combined\2021\Results\Pinus_mugo_classification_selected30_merged.tif"

# 📌 Open raster files
src1 = rasterio.open(raster1)
src2 = rasterio.open(raster2)

# 🔁 Merge using max method (prioritizes higher values like class 1)
merged_array, merged_transform = merge([src1, src2], method='max')

# 📦 Update metadata
out_meta = src1.meta.copy()
out_meta.update({
    "height": merged_array.shape[1],
    "width": merged_array.shape[2],
    "transform": merged_transform,
    "count": merged_array.shape[0],
    "dtype": "uint8",
    "compress": "lzw",
    "BIGTIFF": "YES"
})

# 💾 Write merged raster
with rasterio.open(output_path, "w", **out_meta) as dest:
    dest.write(merged_array.astype("uint8"))

print(f"✅ Merged and saved to: {output_path}")


In [ ]:
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import os

def reproject_to_match_s2_grid(input_path, reference_path, output_path, resampling_method='bilinear'):
    # === Open reference raster (S2 classification) ===
    with rasterio.open(reference_path) as ref:
        ref_crs = ref.crs
        ref_transform = ref.transform
        ref_width = ref.width
        ref_height = ref.height
        ref_profile = ref.profile

    # === Open source raster ===
    with rasterio.open(input_path) as src:
        src_crs = src.crs
        src_transform = src.transform
        src_count = src.count
        src_dtype = src.dtypes[0]
        src_nodata = src.nodata if src.nodata is not None else np.nan

        # Prepare output profile
        profile = ref_profile.copy()
        profile.update({
            'driver': 'GTiff',
            'height': ref_height,
            'width': ref_width,
            'transform': ref_transform,
            'crs': ref_crs,
            'count': src_count,
            'dtype': src_dtype,
            'compress': 'lzw',
            'nodata': src_nodata
        })

        # === Create output raster ===
        with rasterio.open(output_path, 'w', **profile) as dst:
            for i in range(1, src_count + 1):
                print(f"🔁 Reprojecting band {i} of {src_count}...")
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src_transform,
                    src_crs=src_crs,
                    src_nodata=src_nodata,
                    dst_transform=ref_transform,
                    dst_crs=ref_crs,
                    dst_nodata=src_nodata,
                    resampling=getattr(Resampling, resampling_method)
                )

    print("✅ Saved:", output_path)



